# S-Fig 6 — Modality Ablation Bar Chart

ΔAUROC per ablation condition (No BAS, No RESP, etc.) relative to fast-ch baseline.  
**Source**: phase0_v3_abl, phase0_v3, phase0_v3_full  
**Tasks**: main tasks (ablation only covers these)  
**Head**: LSTM

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    """Walk up from CWD until we find a directory containing final_results/."""
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    # Explicit fallback (edit this if auto-detect fails)
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path (notebooks/utils/)
_nb_dir = PAPER_FIGURES / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL, FONT_ANNOT, FONT_BASE, FONT_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

# ── Confirm the workspace root is correct ─────────────────────────────────────
_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND — edit _find_workspace() fallback'}")

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
# NOTE: phase0_v3_abl was trained with LSTM only.
# Changing HEAD to "transformer" or "mean_pool" will produce empty plots
# because no ablation rows exist for those heads.
HEAD   = "lstm"   # ← only valid option for this figure
SPLIT  = "test"
TASKS  = MAIN_TASKS    # ablation only covers main tasks

import pandas as pd
from pathlib import Path

def _load_raw(exp):
    p = WORKSPACE_ROOT / "final_results" / exp / "collected" / "analysis.csv"
    df = pd.read_csv(p)
    if "context_length_min" not in df.columns:
        df["context_length_min"] = df["context_length"].map(
            {"30s": 0.5, "10m": 10.0, "40m": 40.0,
             "80m": 80.0, "120m": 120.0, "240m": 240.0}.get)
    return df

df_abl  = _load_raw("phase0_v3_abl")
df_fast = _load_raw("phase0_v3")
df_full = _load_raw("phase0_v3_full")

abl_heads = sorted(df_abl["head"].unique())
print(f"Ablation heads available: {abl_heads}")
if HEAD not in abl_heads:
    print(f"WARNING: HEAD='{HEAD}' not in ablation data → figure will be empty!")

In [ ]:
N_COLS = 2
N_ROWS = (len(TASKS) + N_COLS - 1) // N_COLS
ROW_H  = 2.4   # inches per row

# Mosaic: 2-2-1 centred layout
labels = [chr(97 + i) for i in range(len(TASKS))]
n_last = len(TASKS) % N_COLS or N_COLS
n_full = len(TASKS) // N_COLS

mosaic = []
for row in range(n_full):
    rl = labels[row * N_COLS : (row + 1) * N_COLS]
    mosaic.append([l for l in rl for _ in range(2)])
if n_last < N_COLS:
    pad = N_COLS - n_last
    ll  = labels[n_full * N_COLS:]
    mosaic.append(["."] * pad + [l for l in ll for _ in range(2)] + ["."] * pad)

fig, axd = plt.subplot_mosaic(mosaic, figsize=(FULL_W, N_ROWS * ROW_H))

for i, (lbl, task) in enumerate(zip(labels, TASKS)):
    ax = axd[lbl]
    col_idx = i % N_COLS   # 0 = left column, 1 = right column
    panels.modality_bar_panel(ax, df_abl, df_fast, df_full, task, head=HEAD, split=SPLIT)
    ax.set_title(TASK_LABEL[task], fontsize=8)
    add_panel_label(ax, f"({lbl})")
    # Hide y-tick labels on right-column panels (left column carries them)
    if col_idx != 0:
        ax.set_yticklabels([])

fig.tight_layout(h_pad=1.0, w_pad=0.8)
plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
save_figure(fig, FINAL_OUT, "sfig6_modality_ablation")
print("Saved →", FINAL_OUT / "sfig6_modality_ablation.pdf")